## Best model notebook

### Group 8 Members
#### Spring Semester 2024-2025
- Alexandre Gon?alves - 20240738
- Br?ulio Damba - 20240007
- Hugo Fonseca - 20240520
- Ricardo Pereira - 20240745
- Victoria Goon - 20240550

## 0 - Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import scale
from sklearn.model_selection import GridSearchCV
import re
import string
import nltk
from sklearn.metrics.pairwise import cosine_distances
from collections import Counter
from nltk.corpus import stopwords
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from nltk.tokenize import TweetTokenizer
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Text extraction 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_distances
from collections import Counter

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from gensim.models import FastText

import scipy.sparse
from scipy import sparse

import contractions
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
import gensim.downloader as api

from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.utils import simple_preprocess

import pickle
from sklearn.utils import resample

from itertools import product

from sentence_transformers import SentenceTransformer

# Deep Learning libraries
from keras.models import Sequential,Model
from keras.layers import Dense, Activation, Dropout, Flatten, Input
from keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, LSTM, Bidirectional, Dropout, Flatten, GRU
from tensorflow.keras.optimizers import Adam
from transformers import AutoTokenizer, AutoModel
import torch

# Set pd options to display all columns and rows
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 30)
pd.set_option('display.max_colwidth', None)  

import glob
from scipy.sparse import load_npz

from tensorflow.keras.layers import Layer
import tensorflow as tf
from tensorflow.keras.layers import Input, Bidirectional, GRU, Flatten, Dense

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.utils import to_categorical
from sklearn.base import clone


from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForSequenceClassification,
    BartConfig,
)
import optuna

import json

# Download required resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

### Import the encoded files

In [ ]:
# Define the base directory (where the notebook is)
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

# Construct full paths to the CSV files
train_path = os.path.join(BASE_DIR, "data", "train.csv")
test_path = os.path.join(BASE_DIR, "data", "test.csv")

# Load the datasets
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [ ]:
folder = '.'  # Use '.' if running in the current directory, or the path to your folder

# List all .npy and .npz files
npy_files = glob.glob(os.path.join(folder, '*.npy'))
npz_files = glob.glob(os.path.join(folder, '*.npz'))

In [ ]:
npy_embeddings = {}

for f in npy_files:
    name = os.path.splitext(os.path.basename(f))[0]
    npy_embeddings[name] = np.load(f, allow_pickle=True)

In [ ]:
npz_embeddings = {}

for f in npz_files:
    name = os.path.splitext(os.path.basename(f))[0]
    try:
        npz_embeddings[name] = load_npz(f)
    except Exception as e:
        print(f"Warning: Failed to load '{f}' as a sparse matrix. Error: {e}")
        try:
            npz_embeddings[name] = np.load(f)
            print(f"Loaded '{f}' as dense array (np.load).")
        except Exception as e2:
            print(f"Failed to load '{f}' as dense array too. Skipping. Error: {e2}")


In [ ]:
# Using stratify to maintain the distribution of classes in the train, validation, and test sets 
# As our dataset is quite small, we use 80% for training, and split the remaining 20% into validation and test sets (10% each).

train_df, val_test_df = train_test_split(df_train, test_size=0.2, stratify=df_train['label'], random_state=42)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df['label'], random_state=42)

In [ ]:
y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

In [ ]:
# Load from file
with open("best_bart_params.json", "r") as f:
    best_params = json.load(f)

print("Best parameters loaded:", best_params)

## 1 - Hyperparameter tuning BART

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Tokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base", use_fast=True)

# Dataset construction
train_df = Dataset.from_dict({
    "text_no_lemma_stem_with_stopwords": npy_embeddings["train_text_no_lemma_stem_with_stopwords"],
    "label": y_train.tolist()
})
val_df = Dataset.from_dict({
    "text_no_lemma_stem_with_stopwords": npy_embeddings["val_text_no_lemma_stem_with_stopwords"],
    "label": y_val.tolist()
})
test_df = Dataset.from_dict({
    "text_no_lemma_stem_with_stopwords": npy_embeddings["test_text_no_lemma_stem_with_stopwords"],
    "label": y_test.tolist()
})


In [ ]:
# Tokenization 
def tokenize_dataset(dataset):
    return dataset.map(
        lambda examples: tokenizer(
            examples["text_no_lemma_stem_with_stopwords"],
            padding="max_length",
            truncation=True,
            max_length=64
        ),
        batched=True
    )
    
# Metric for optimization
def compute_metrics(pred):
    try:
        logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
        preds = np.argmax(logits, axis=1)
        labels = pred.label_ids
        return {"macro_f1": f1_score(labels, preds, average="macro")}
    except Exception as e:
        print(f"Metric computation error: {e}")
        return {"macro_f1": 0.0}

In [ ]:
# Optuna search space
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_categorical("learning_rate", [1e-5, 2e-5, 3e-5, 5e-5]),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "num_train_epochs": trial.suggest_categorical("num_train_epochs", [3, 4, 5]),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 0.01]),
        "warmup_steps": trial.suggest_categorical("warmup_steps", [0, 100, 500]),
        "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2]),
        "dropout": trial.suggest_categorical("dropout", [0.1, 0.2]),
    }

# Model init
def model_init(trial):
    dropout = trial.suggest_categorical("dropout", [0.1, 0.2])
    config = BartConfig.from_pretrained("facebook/bart-base", num_labels=3, attention_dropout=dropout, dropout=dropout)
    return BartForSequenceClassification.from_pretrained("facebook/bart-base", config=config)

# TrainingArguments
def build_training_args(trial):
    return TrainingArguments(
        output_dir="./results_bart_tune",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=trial.suggest_categorical("learning_rate", [1e-5, 2e-5, 3e-5, 5e-5]),
        per_device_train_batch_size=trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        num_train_epochs=trial.suggest_categorical("num_train_epochs", [3, 4, 5]),
        weight_decay=trial.suggest_categorical("weight_decay", [0.0, 0.01]),
        warmup_steps=trial.suggest_categorical("warmup_steps", [0, 100, 500]),
        gradient_accumulation_steps=trial.suggest_categorical("gradient_accumulation_steps", [1, 2]),
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none"
    )

def objective(trial):
    train_dataset = tokenize_dataset(train_df).rename_column("label", "labels")
    val_dataset = tokenize_dataset(val_df).rename_column("label", "labels")

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    trainer = Trainer(
        model_init=lambda: model_init(trial),
        args=build_training_args(trial),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    eval_result = trainer.evaluate()
    return eval_result["eval_macro_f1"]

In [ ]:
# Run Optuna search
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# Display best hyperparameters
print("Best hyperparameters found:")
print(study.best_params)

In [ ]:
# Save to file
with open("best_bart_params.json", "w") as f:
    json.dump(best_params, f)

print("Best parameters saved to best_bart_params.json")

## 2 - BART with best hyperparameters

In [ ]:
# Tokenize 
train_dataset = tokenize_dataset(train_df).rename_column("label", "labels")
val_dataset = tokenize_dataset(val_df).rename_column("label", "labels")
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

best_config = BartConfig.from_pretrained(
    "facebook/bart-base",
    num_labels=3,
    attention_dropout=best_params["dropout"],
    dropout=best_params["dropout"],
)
best_model = BartForSequenceClassification.from_pretrained("facebook/bart-base", config=best_config).to(device)

# Set training arguments
best_args = TrainingArguments(
    output_dir="./results_bart_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["per_device_train_batch_size"],
    num_train_epochs=best_params["num_train_epochs"],
    weight_decay=best_params["weight_decay"],
    warmup_steps=best_params["warmup_steps"],
    gradient_accumulation_steps=best_params["gradient_accumulation_steps"],
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

# Final Trainer
final_trainer = Trainer(
    model=best_model,
    args=best_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Train on best configuration
final_trainer.train()

# Predict and evaluate
preds = final_trainer.predict(val_dataset)
logits = preds.predictions[0] if isinstance(preds.predictions, tuple) else preds.predictions
y_pred = np.argmax(logits, axis=1)
y_true = np.array(val_df["label"])

print(classification_report(y_true, y_pred, digits=4))

## 3 - Dealing with Class Imbalance

### 3.2 - BART with best hyperparameters with random undersampling

In [ ]:
# Tokenizer function
text_key = "text_no_lemma_stem_with_stopwords"

def tokenize_dataset(dataset, text_key):
    return dataset.map(lambda e: tokenizer(e[text_key], padding="max_length", truncation=True, max_length=64), batched=True)

# Create original training DataFrame
train_texts = npy_embeddings["train_text_no_lemma_stem_with_stopwords"]
train_labels = y_train
train_df_raw = pd.DataFrame({
    "text": train_texts,
    "label": train_labels
})

# Random Undersampling
min_class_size = train_df_raw['label'].value_counts().min()
df_undersampled = pd.concat([
    resample(group, replace=False, n_samples=min_class_size, random_state=42)
    for _, group in train_df_raw.groupby('label')
])
df_undersampled = df_undersampled.sample(frac=1, random_state=42).reset_index(drop=True)

# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(df_undersampled.rename(columns={"text": text_key}))
val_dataset = Dataset.from_dict({
    text_key: npy_embeddings["val_text_no_lemma_stem_with_stopwords"],
    "label": y_val.tolist()
})

# Tokenize 
train_dataset = tokenize_dataset(train_dataset, text_key).rename_column("label", "labels")
val_dataset = tokenize_dataset(val_dataset, text_key).rename_column("label", "labels")
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Load model with best config from Optuna 
best_config = BartConfig.from_pretrained(
    "facebook/bart-base",
    num_labels=3,
    attention_dropout=best_params["dropout"],
    dropout=best_params["dropout"]
)
best_model = BartForSequenceClassification.from_pretrained("facebook/bart-base", config=best_config).to(device)

best_args = TrainingArguments(
    output_dir="./results_bart_undersample",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["per_device_train_batch_size"],
    num_train_epochs=best_params["num_train_epochs"],
    weight_decay=best_params["weight_decay"],
    warmup_steps=best_params["warmup_steps"],
    gradient_accumulation_steps=best_params["gradient_accumulation_steps"],
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

final_trainer = Trainer(
    model=best_model,
    args=best_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

final_trainer.train()

preds = final_trainer.predict(val_dataset)
logits = preds.predictions[0] if isinstance(preds.predictions, tuple) else preds.predictions
y_pred = np.argmax(logits, axis=1)
y_true = np.array(y_val)

print(classification_report(y_true, y_pred, digits=4))

As we can see, applying the random undersampling didnt improve the overall performance of the BART transformer, so we will evaluate performance on test, and later on making the predictions using the **BART transformer with the best parameters**

## 4 - Checking BART with best hyperparameters on test

In [ ]:
label_map = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

df_test_raw = test_df.to_pandas().reset_index(drop=True)

# Track valid rows
valid_indices = []
valid_texts = []

for i, text in enumerate(df_test_raw["text_no_lemma_stem_with_stopwords"]):
    if isinstance(text, str) and text.strip():
        valid_indices.append(i)
        valid_texts.append(text)

# Tokenize only valid rows
valid_dataset = Dataset.from_pandas(pd.DataFrame({
    "text_no_lemma_stem_with_stopwords": valid_texts
}))
valid_dataset = tokenize_dataset(valid_dataset)
valid_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

test_preds = final_trainer.predict(valid_dataset)
test_logits = test_preds.predictions[0] if isinstance(test_preds.predictions, tuple) else test_preds.predictions
test_y_pred_valid = np.argmax(test_logits, axis=1)

# Reconstruct full-length predictions
test_y_pred = [None] * len(df_test_raw)
for idx, pred in zip(valid_indices, test_y_pred_valid):
    test_y_pred[idx] = pred

df_test_raw["true_label"] = y_test.values
df_test_raw["pred_label"] = test_y_pred

df_eval = df_test_raw[df_test_raw["pred_label"].notna()].copy()
df_eval["pred_label"] = df_eval["pred_label"].astype(int)

test_y_true = df_eval["true_label"]
test_y_pred = df_eval["pred_label"]

print("\nClassification Report (Test Set):")
print(classification_report(test_y_true, test_y_pred, target_names=label_map.values(), digits=4))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Generate the confusion matrix
cm = confusion_matrix(test_y_true, test_y_pred)

class_names = ["Bearish", "Bullish", "Neutral"]

# Plot the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", ax=ax, values_format='d')

plt.title("Confusion Matrix - Test Set")
plt.show()

The confusion matrix reveals that the BART model performs best at identifying Neutral and Bullish tweets, correctly classifying approximately 90% of Neutral and 78% of Bullish samples. However, the model shows some difficulty distinguishing Bearish tweets, with a notable portion being misclassified as Neutral. This confusion might stem from the linguistic similarities between Neutral and Bearish expressions, especially in financial contexts where subtle phrasing can blur the sentiment. Additionally, if the training data had fewer Bearish examples, the model may not have learned enough distinctive patterns for that class.

In [ ]:
# Identify misclassified samples 
misclassified = df_eval[test_y_true != test_y_pred]

# Show 5 misclassified examples per true class
for label_id in sorted(test_y_true.unique()):
    print(f"\nMisclassified examples for true class '{label_map[label_id]}':\n")
    
    subset = misclassified[misclassified["true_label"] == label_id]
    sample = subset.sample(n=min(5, len(subset)), random_state=42)
    
    for _, row in sample.iterrows():
        print(f"- Text: {row['text_no_lemma_stem_with_stopwords']}")
        print(f"  True: {label_map[row['true_label']]}, Predicted: {label_map[row['pred_label']]}\n")

## 5 - Making predictions

In [ ]:
df_test = df_test.rename(columns={"text": "text_no_lemma_stem_with_stopwords"})

hf_test_dataset = Dataset.from_pandas(df_test)

# Tokenize the test dataset
hf_test_dataset = hf_test_dataset.map(
    lambda examples: tokenizer(
        examples["text_no_lemma_stem_with_stopwords"],
        padding="max_length",
        truncation=True,
        max_length=64
    ),
    batched=True
)

# Prepare for model input
hf_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

preds = final_trainer.predict(hf_test_dataset)
logits = preds.predictions[0] if isinstance(preds.predictions, tuple) else preds.predictions
test_predictions = np.argmax(logits, axis=1)

# Export predictions to CSV
df_submission = pd.DataFrame({
    "id": df_test["id"],  
    "label": test_predictions
})
df_submission.to_csv("pred_bart.csv", index=False)

print("Test predictions saved to pred_bart.csv")